# MATH840 — Lab 2: Decomposing your series

**Week 2 | graded assignment 1 of 7 | 8 points | due at the end of the session**

You have a series. This week you find out what is in it, and commit to a claim about it:
which of the four simple methods will be hardest to beat. Next week you test that claim.

**Keep the section headings below.** They map one-to-one onto the marking rubric:

| Section | Criteria | Points |
|---|---|---|
| 1 | Data preparation | 1 |
| 2 | EDA and visualisation | 4 |
| 3 | Implementation | 2 |
| 4 | Code quality and reproducibility | 1 |

Submit `SURNAME_lab02.pdf`, `SURNAME_lab02.ipynb` and `SURNAME_lab02_summary.csv` to Moodle
before the end of the session. A valid partial submission scores; a perfect missing one
does not.

## 0. Setup

Given — run it and move on.

In [ ]:
!pip install -q statsforecast utilsforecast coreforecast

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf
from coreforecast.scalers import boxcox, boxcox_lambda

pd.set_option("display.width", 120)
print("pandas", pd.__version__)

In [ ]:
SURNAME = ""   # <-- for your submission file names
MY_ID   = ""   # <-- the SAME identifier you used in Week 1

if not SURNAME.strip() or not MY_ID.strip():
    raise ValueError("Set SURNAME and MY_ID before running the rest.")

The assignment function from Week 1, unchanged. The same identifier gives you the same
series — **do not switch datasets**, weeks 2 and 3 are two halves of one exercise.

In [ ]:
BASE = ("https://raw.githubusercontent.com/Aranaur/aranaur.rbind.io/"
        "main/lectures/kse/MATH840/26autumn/data")

POOL = (
    [{"file": "aus_production.csv", "column": c, "freq": "QS", "season": 4,
      "label": f"Australian production: {c}"}
     for c in ["Beer", "Tobacco", "Bricks", "Cement", "Electricity", "Gas"]]
    + [{"file": "tourism.csv", "state": s, "purpose": p, "freq": "QS", "season": 4,
        "label": f"Tourism: {p} trips to {s}"}
       for s in ["South Australia", "Northern Territory", "Western Australia",
                 "Victoria", "New South Wales", "Queensland", "ACT", "Tasmania"]
       for p in ["Business", "Holiday", "Other", "Visiting"]]
    + [{"file": "ansett.csv", "route": r, "class": c, "freq": "W-MON", "season": 52,
        "label": f"Ansett passengers: {r}, {c} class"}
       for r in ["MEL-SYD", "MEL-ADL", "SYD-BNE", "MEL-BNE", "ADL-PER", "MEL-PER"]
       for c in ["Business", "Economy"]]
)


def assign_dataset(student_id: str) -> dict:
    digest = hashlib.sha256(student_id.strip().encode("utf-8")).hexdigest()
    return POOL[int(digest, 16) % len(POOL)]


def load_my_series(spec: dict) -> pd.DataFrame:
    df = pd.read_csv(f"{BASE}/{spec['file']}")
    if spec["file"] == "aus_production.csv":
        out = df[["ds", spec["column"]]].rename(columns={spec["column"]: "y"})
    elif spec["file"] == "tourism.csv":
        sub = df[(df["State"] == spec["state"]) & (df["Purpose"] == spec["purpose"])]
        out = sub.groupby("ds", as_index=False)["y"].sum()
    else:
        sub = df[(df["Airports"] == spec["route"]) & (df["Class"] == spec["class"])]
        out = sub[["ds", "y"]].copy()
    out["ds"] = pd.to_datetime(out["ds"])
    return out.sort_values("ds").reset_index(drop=True)


mine = assign_dataset(MY_ID)
raw = load_my_series(mine)
print(mine["label"], "|", mine["freq"], "| season length", mine["season"])
raw.head()

## 1. Data preparation

*1 point.* Put the series on a regular time grid and describe it.

- Resolve duplicated timestamps if there are any — and say in one sentence what you decided
  and why.
- Fill the calendar so that gaps become visible missing values instead of absent rows.
- State the frequency, the span, and how many complete seasonal cycles you have.

You did all of this in Week 1; it should take ten minutes.

In [ ]:
# TODO: duplicates, regular grid, and the four facts about your series.
# End with a clean frame called `s` with columns ds, y.

s = raw.copy()

**Your description.** *(frequency, span, complete cycles, what you did about duplicates and gaps)*

→

## 2. EDA and visualisation

*4 points — the heaviest section.* Every plot needs a sentence saying what you see in it.
A plot with no reading attached earns nothing.

### 2.1 Graphics

Time plot, seasonal plot, seasonal subseries plot, ACF. Name the patterns: trend,
seasonality, cycles, level shifts, outliers.

In [ ]:
# TODO: time plot

In [ ]:
# TODO: seasonal plot

In [ ]:
# TODO: seasonal subseries plot

In [ ]:
# TODO: ACF

**What you see.**

→

### 2.2 Variance

Does the size of the seasonal swing grow with the level of the series? If it does, an
additive decomposition is the wrong model for it.

Estimate a Box-Cox $\lambda$, plot the transformed series next to the original, and then
**decide**. A defensible "no transformation needed" earns full marks; an unexplained
transformation does not.

```python
lam = boxcox_lambda(y, method="loglik", season_length=mine["season"])
y_transformed = boxcox(y, lam)
```

**If your series contains zeros:** `method="loglik"` refuses non-positive values, `method="guerrero"` may return $\lambda = 0$ (the log transform, undefined at zero), and a multiplicative decomposition refuses them too. That is information about your series, not an obstacle — find out where the zeros come from, and say what you concluded.


In [ ]:
# TODO: estimate lambda, plot original vs transformed, decide.
# Set `work` to the version of the series you will decompose below.

**Your decision and why.**

→

### 2.3 Decomposition

Two of them, on the version you settled on: classical (`seasonal_decompose`) and STL
(`STL`, with a stated choice of `seasonal` and `robust`). Then compare: where do they
disagree, and which do you trust more here?

In [ ]:
# TODO: classical decomposition

In [ ]:
# TODO: STL decomposition

**Reading the components.**

Not "the trend goes up" — something a colleague could act on. *The seasonal component is
stable until 2008 and shifts afterwards. The remainder is flat except for three spikes.
The trend flattens in the last two years.*

→

## 3. Implementation

*2 points.*

### 3.1 Strength of trend and seasonality

$$
F_T = \max\left(0,\; 1 - \frac{\operatorname{Var}(R_t)}{\operatorname{Var}(T_t + R_t)}\right)
\qquad
F_S = \max\left(0,\; 1 - \frac{\operatorname{Var}(R_t)}{\operatorname{Var}(S_t + R_t)}\right)
$$

In [ ]:
def strengths(stl_fit) -> tuple[float, float]:
    """Return (F_T, F_S) from a fitted STL decomposition."""
    # TODO
    raise NotImplementedError


F_T, F_S = strengths(...)   # TODO: pass your fitted STL
print(f"F_T = {F_T:.2f}   F_S = {F_S:.2f}")

### 3.2 Seasonally adjusted series

Produce it and plot it. Say what it is useful for — and what it hides.

In [ ]:
# TODO: seasonally adjusted series

**What it is for, and what it hides.**

→

### 3.3 Your prediction

Of the four simple methods — `mean`, `naive`, `drift`, `snaive` — which will be **hardest
to beat** on your series?

Argue for it in three or four sentences using $F_T$, $F_S$ and what the decomposition
showed you. The prediction is marked on the **reasoning**, not on being right: a
well-argued wrong answer earns full marks, and *"snaive, because it usually wins"* earns
nothing even though it is usually true.

In [ ]:
PREDICTION = ""   # one of: "mean", "naive", "drift", "snaive"

assert PREDICTION in {"mean", "naive", "drift", "snaive"}, "pick one of the four"

**Your argument.**

→

## 4. Code quality and reproducibility

*1 point.* Before you export:

- Restart the kernel and run everything top to bottom. It must finish without errors.
- Delete dead cells and leftover experiments.
- Check that every number in your text comes from a cell, not from your memory of an
  earlier run.

## 5. Submit

Run the cell below, then export and upload **three files** to the Week 2 activity on
Moodle before the end of the session:

- `SURNAME_lab02.pdf` — `File → Print → Save as PDF`
- `SURNAME_lab02.ipynb` — `File → Download → Download .ipynb`
- `SURNAME_lab02_summary.csv` — written by the cell below

In [ ]:
summary = pd.DataFrame([{
    "student_id": MY_ID,
    "series": mine["label"],
    "F_T": round(float(F_T), 2),
    "F_S": round(float(F_S), 2),
    "prediction": PREDICTION,
}])

name = f"{SURNAME.strip().upper()}_lab02_summary.csv"
summary.to_csv(name, index=False)
print(summary.to_string(index=False))

try:
    from google.colab import files
    files.download(name)
except ImportError:
    print(f"\nNot in Colab — {name} is saved next to this notebook.")